In [2]:
import requests

import numpy as np
import pandas as pd
import plotly.express as px
import folium
import shutil
import glob
import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [3]:
OUTPUT_DIR = "../data/JC"
output_dir = Path(OUTPUT_DIR)

file_names = glob.glob(f'{output_dir}/*.csv')

dfs = []
cols = []
for file_name in file_names:
    df = pd.read_csv(file_name)
    print(df.columns, 2*"||",len(df.columns))

    cols.append(list(df.columns))
    dfs.append(df)

citibike_df = pd.concat(dfs)
citibike_df.head()

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      d

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_minutes,date,month,month_name,day_of_week,hour,season
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
18,hour,1954797,0.648401
17,day_of_week,1954797,0.648401
19,season,1954797,0.648401
16,month_name,1954797,0.648401
14,date,1954797,0.648401
15,month,1954797,0.648401
13,ride_duration_minutes,1954797,0.648401
7,end_station_id,9795,0.003249
6,end_station_name,7315,0.002426
10,end_lat,6869,0.002278


In [5]:
citibike_df['started_at'] = pd.to_datetime(citibike_df['started_at'],errors="coerce")
citibike_df['ended_at'] = pd.to_datetime(citibike_df['ended_at'],errors="coerce")

In [6]:
# ride duration

citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

In [7]:
#removes impossible rides: negative duration or longer than 24 hours.

citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 0) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
]

In [8]:
citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)

citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

In [9]:
citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [10]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [11]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2025-01-16 17:50:49.136,2025-01-16,2025-01,January,Thursday,17,Winter
1,2025-01-31 06:10:41.818,2025-01-31,2025-01,January,Friday,6,Winter
2,2025-01-09 16:42:50.213,2025-01-09,2025-01,January,Thursday,16,Winter
3,2025-01-21 16:14:14.398,2025-01-21,2025-01,January,Tuesday,16,Winter
4,2025-01-30 16:38:18.840,2025-01-30,2025-01,January,Thursday,16,Winter


In [12]:
citibike_df.info()

<class 'pandas.DataFrame'>
Index: 3007860 entries, 0 to 1059999
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   ride_id                str           
 1   rideable_type          str           
 2   started_at             datetime64[us]
 3   ended_at               datetime64[us]
 4   start_station_name     str           
 5   start_station_id       str           
 6   end_station_name       str           
 7   end_station_id         str           
 8   start_lat              float64       
 9   start_lng              float64       
 10  end_lat                float64       
 11  end_lng                float64       
 12  member_casual          str           
 13  ride_duration_minutes  float64       
 14  date                   object        
 15  month                  str           
 16  month_name             str           
 17  day_of_week            str           
 18  hour                   int32         


In [13]:
citibike_df.to_csv("../data/JC/JC2025_Enriched.csv", index = False)